# Phase 2: Data Enrichment via Web Scraping

## Objective
The `diag_1` column stores each patient's main diagnosis as an ICD-9 code, like `428` or `250.8`. A doctor knows that `428` means heart failure, but a hospital administrator just sees a number. In this phase I fix that by scraping the disease names from a public ICD-9 website and adding them to the dataset, so anyone reading the results can understand them.

## Why this matters for VHN
- The CMIO asked us to explain *why* readmissions happen, not just report what happened. That is hard to do when the diagnoses are only numbers.
- Readable labels like "Heart failure" make the later phases (EDA, the risk score and the board report) much easier to follow.
- It also shows how an internal dataset can be improved with outside reference data.

## What I will do in this phase

| Step | Task | Result |
|------|------|--------|
| 2.1 | Find the 20 most common primary diagnosis codes | `top20_codes` list |
| 2.2 | Scrape their descriptions using `requests` and `BeautifulSoup` | ICD-9 lookup table |
| 2.3 | Add the descriptions to the main dataset | `Primary_Diagnosis_Desc` column |

## Input and output
- **Input:** `df_cleaned` from Phase 1 (`?` converted to NaN, `weight` dropped, deceased patients and duplicates removed).
- **Output:** the same `df_cleaned` with one new column, `Primary_Diagnosis_Desc`.

## 2.1 Target Identification: Top 20 Primary Diagnoses

### What I am doing
I count how often each ICD-9 code appears in `diag_1` and select the **20 most frequent** codes. These become the only codes that will be looked up online.

### Why only the top 20?
1. **Efficiency:** The dataset contains hundreds of distinct `diag_1` codes. Looking up every one would take a long time, because the scraper must wait one second between requests.
2. **Ethical scraping:** Fewer requests means less load on the host server and a lower risk of the IP address being blocked.
3. **Pareto principle:** A small number of conditions usually account for a large share of all hospital encounters. Focusing on them gives the best insight per request.
4. **Project requirement:** The specification explicitly asks for the top 20 codes as the scraping targets.

### Expected outcome
A ranked table of the 20 codes, and a Python list `top20_codes` which is passed to the scraper in step 2.2.

In [ ]:
# 2.1 Target Identification: Top 20 primary diagnosis codes (diag_1)

TOP_N = 20

top20_df = (df_cleaned['diag_1'].value_counts()
            .head(TOP_N)
            .rename_axis('diag_1')
            .reset_index(name='encounters'))
top20_df.index = top20_df.index + 1 
top20_df.index.name = 'rank'

print(f"Distinct diag_1 codes in the cleaned data: {df_cleaned['diag_1'].nunique():,}")
display(top20_df)

# list that will be passed to the scraper in step 2.2
top20_codes = top20_df['diag_1'].tolist()
print("Scraping targets:", top20_codes)

### Findings from Step 2.1
- The most frequent primary diagnosis is **428** with 6,735 encounters, followed closely by **414** with 6,554. After these two, the counts drop quite a lot: **786** is third with 4,016 encounters.
- The top 20 mixes diabetes codes (`250.8`, `250.6`) with other conditions that often occur alongside diabetes, such as heart-related codes (428, 414, 410, 427) and lung-related codes (486, 491).
- The counts fall steadily down the list. From the 7th code onwards they are fairly close to each other, ranging from about 1,000 to 2,300 encounters.
- Some codes are formatted differently from the others. For example, `38` has lost its leading zero (it should be `038`) and `V57` starts with a letter. I will need to handle this when building the scraper in step 2.2.

**Conclusion:** These 20 codes are my scraping targets for the next step.

## 2.2 Scraper Architecture

### What I am doing
I build a scraper that looks up the description of each of the 20 target codes on the public ICD-9 website `http://icd9.chrisendres.com/`, using `requests` to download the pages and `BeautifulSoup` to read them.

### How the scraper works
The website is organised like a book: **chapter → section → code**. For example, code `428` is inside the "Circulatory System" chapter, then inside the "Other Forms of Heart Disease (420-429)" section. The scraper starts at the main index and keeps following the link whose number range contains the code, until it finds the row for that code. For codes with a decimal part (like `250.8`), it goes one level deeper into the page of the main code (`250`).

### Decisions I made
- **Browsing instead of the search box:** The site's search link did not give reliable results when I tested it, so the scraper follows the index pages instead.
- **Fixing the code format:** Some codes lost their leading zeros in the CSV (for example `38` should be `038`, Septicemia), so I pad numeric codes to 3 digits before looking them up. `V` and `E` codes are left as they are.
- **Ethical scraping:** One second pause (`time.sleep(1)`) after every request, a custom `User-Agent`, a timeout, and a maximum of 3 retries.
- **Error handling:** If one code fails, the scraper reports it and moves on, so one problem does not stop the whole run.
- **Saving results:** The results are saved to a CSV file, so I do not have to scrape again every time I run the notebook.

### Expected outcome
A lookup table with one description for each of the 20 codes. I will use it in step 2.3.

In [ ]:
# 2.2 Scraper: functions 

import re
from urllib.parse import urljoin

BASE_URL = "http://icd9.chrisendres.com/"
START_URL = BASE_URL + "index.php?action=contents"
HEADERS = {"User-Agent": "Mozilla/5.0 (Academic project - IJSE ITS2122)"}
REQUEST_DELAY = 1     
MAX_RETRIES = 3
TIMEOUT = 10


CODE_ROW = re.compile(r"^([EV]?\d{2,3}(?:\.\d+)?)\s+(.+)$")

RANGE_END = re.compile(r"\(([EV]?\d+)(?:-([EV]?\d+))?\)\s*$")

page_cache = {}      


def normalize_icd9(code):
    """Pad numeric codes to 3 digits: '38' -> '038'. E and V codes stay the same."""
    code = str(code).strip()
    if code[0] in ("E", "V"):
        return code
    whole, _, decimal = code.partition(".")
    whole = whole.zfill(3)
    return f"{whole}.{decimal}" if decimal else whole


def split_code(code):
    """'V57' -> ('V', 57), '428' -> ('', 428)"""
    letter = code[0] if code[0] in ("E", "V") else ""
    return letter, int(float(code[len(letter):]))


def in_range(base, low, high):
    """Is the base code (e.g. '428') inside a section range like 420-429?"""
    b_letter, b_num = split_code(base)
    l_letter, l_num = split_code(low)
    h_letter, h_num = split_code(high)
    return b_letter == l_letter == h_letter and l_num <= b_num <= h_num


def fetch_page(url):
    """Download a page (with retries and a delay). Returns the HTML or None."""
    if url in page_cache:
        return page_cache[url]
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            time.sleep(REQUEST_DELAY)         
            page_cache[url] = response.text
            return response.text
        except requests.exceptions.RequestException as error:
            print(f"   attempt {attempt}/{MAX_RETRIES} failed: {error}")
            time.sleep(REQUEST_DELAY * attempt)
    return None


def parse_rows(html, page_url):
    """Return a list of (first line of text, link or None) for every table row."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for tr in soup.find_all("tr"):
        cells = tr.find_all("td", recursive=False)
        if not cells:
            continue
        cell = cells[-1]                        
        text = cell.get_text("\n", strip=True).replace("\xa0", " ")
        if not text:
            continue
        first_line = text.split("\n")[0].strip()
        link = cell.find("a", href=re.compile("action=child"))
        rows.append((first_line, urljoin(page_url, link["href"]) if link else None))
    return rows


def lookup_icd9(code):
    """Find the description of one ICD-9 code. Returns the text, or None if not found."""
    code = normalize_icd9(code)
    base = code.split(".")[0]
    url, visited = START_URL, set()

    for _ in range(8):                          
        visited.add(url)
        html = fetch_page(url)
        if html is None:
            return None
        rows = parse_rows(html, url)

        for text, link in rows:
            match = CODE_ROW.match(text)
            if match and match.group(1) == code:
                return match.group(2).strip()

        if "." in code and len(code.split(".")[1]) == 1:
            for text, link in rows:
                match = CODE_ROW.match(text)
                if match and match.group(1) == code + "0":
                    return match.group(2).strip()

        next_url = None
        for text, link in rows:
            if not link or link in visited:
                continue
            match = CODE_ROW.match(text)
            if match:
                if code.startswith(match.group(1) + "."):    
                    next_url = link
                    break
            else:
                section = RANGE_END.search(text)
                if section and in_range(base, section.group(1), section.group(2) or section.group(1)):
                    next_url = link
                    break
        if next_url is None:
            return None
        url = next_url
    return None


#  quick test with 1 code before the full run
print("Test 428   ->", lookup_icd9("428"))
print("Test 250.8 ->", lookup_icd9("250.8"))

In [ ]:
# 2.2 Scraper: run on the top 20 codes - the results are saved to a CSV file

CACHE_FILE = "icd9_top20_descriptions.csv"


def scrape_top_codes(codes):
    results = []
    for i, code in enumerate(codes, start=1):
        print(f"[{i}/{len(codes)}] Looking up {code} ...")
        try:
            description = lookup_icd9(code)
        except Exception as error:             
            print(f"   error for {code}: {error}")
            description = None
        results.append({"diag_1": code, "icd9_description": description})
    return pd.DataFrame(results)


if os.path.exists(CACHE_FILE):
    icd9_lookup = pd.read_csv(CACHE_FILE, dtype={"diag_1": str})
    print(f"Loaded saved results from {CACHE_FILE}")
else:
    icd9_lookup = scrape_top_codes(top20_codes)
    icd9_lookup.to_csv(CACHE_FILE, index=False)
    print(f"Results saved to {CACHE_FILE}")

found = icd9_lookup["icd9_description"].notna().sum()
print(f"\nDescriptions found: {found} out of {len(icd9_lookup)}")

failed = icd9_lookup[icd9_lookup["icd9_description"].isna()]["diag_1"].tolist()
if failed:
    print("Codes without a description:", failed)

display(icd9_lookup)

In [ ]:
# 2.2 Scraper: run on the top 20 codes 

results = []
for i, code in enumerate(top20_codes, start=1):
    print(f"[{i}/{len(top20_codes)}] Looking up {code} ...")
    try:
        description = lookup_icd9(code)
    except Exception as error:             
        print(f"   error for {code}: {error}")
        description = None
    results.append({"diag_1": code, "icd9_description": description})

icd9_lookup = pd.DataFrame(results)

found = icd9_lookup["icd9_description"].notna().sum()
print(f"\nDescriptions found: {found} out of {len(icd9_lookup)}")

failed = icd9_lookup[icd9_lookup["icd9_description"].isna()]["diag_1"].tolist()
if failed:
    print("Codes without a description:", failed)

display(icd9_lookup)

### Findings from Step 2.2
- The scraper found descriptions for **all 20** codes.
- For example, `428` is **Heart failure** and `414` is **Other forms of chronic ischemic heart disease**.
- The scraper pauses one second after every request, so it takes a minute or two to run.

**Conclusion:** The lookup table is ready to be merged into the main dataset.